# **Hindawi · a large language model, built from Hindi**

This notebook takes llama.cpp with its source in your language (Urdu, Hindi, Telugu and others), lowers it through the original Hindawi pipeline, builds it with CUDA, and runs a large language model on Colab's GPU.

**How to use it:** Runtime → Change runtime type → a GPU. Press ▶ on Installation, then on *Choose a model*, then on *Ask it*.

*Evidence:* written and checked by Parashakti's run of llama.cpp; its first recorded run on Colab is still to come.

# Installation

In [ ]:
#@title **Installation: the tools, and llama.cpp in your language** (press ▶ once; ten to twenty minutes on a GPU) { vertical-output: true }
#@markdown Choose the language the source should be in. For a GPU: Runtime → Change runtime type → T4 GPU (or larger).
language = "hindi" #@param ["arabic", "assamese", "awadhi", "balochi", "bengali", "bhojpuri", "bodo", "chhattisgarhi", "dari", "dogri", "gujarati", "hausa_ajami", "hindi", "kannada", "kashmiri", "kashmiri_dev", "konkani", "kurdish_sorani", "magahi", "maithili", "malay_jawi", "malayalam", "mandaic", "manipuri_beng", "marathi", "marwari", "nabataean", "nepali", "newari", "odia", "pali", "pashto", "persian", "prakrit", "punjabi", "punjabi_shahmukhi", "rajasthani", "samaritan", "sanskrit", "saraiki", "sindhi", "sindhi_dev", "tagalog", "tamil", "telugu", "tulu", "urdu", "uyghur"]
import os, shutil, subprocess, pathlib, re
def sh(c, show=True):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    if show and r.returncode: print((r.stdout + r.stderr)[-1500:])
    return r.returncode
if not os.path.exists("/usr/local/bin/hin"):
    print("Fetching Hindawi's prebuilt tools ...")
    if sh("cd /content && curl -fsSL -o t.tgz https://github.com/hindawiai/hindawi2026.10/releases/latest/download/hindawi-tools-linux-x86_64.tar.gz && mkdir -p hindawi && tar -xzf t.tgz -C hindawi && rm t.tgz", False):
        sh("apt-get -qq install -y gawk flex bison >/dev/null; cd /content && git clone -q --depth 1 https://github.com/hindawiai/hindawi2026.10 hindawi && cd hindawi && bash scripts/build.sh >/dev/null 2>&1")
    sh("bash /content/hindawi/scripts/colab_env.sh")
H = "/content/hindawi"; os.environ["ROMENAGRI"] = H + "/.build/chintamani/Romenagri"; os.environ["PATH"] = os.environ["ROMENAGRI"] + ":" + os.environ["PATH"]
src = f"/content/llama-{language}"
if not os.path.isdir(src):
    print(f"Fetching llama.cpp in {language} from github.com/hmnsq ...")
    if sh(f"git clone -q --depth 1 -b lang/{language} https://github.com/hmnsq/llama.cpp {src}", False):
        print("Not published in that language yet: ingesting the upstream into it here ...")
        sh("rm -rf /content/up /content/n && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/up")
        sh(f"python3 {H}/lang/hindawi/burp/burp.py ingest /content/up --langs {language} --out /content/n --translators glossary --keep '.*'")
        sh(f"cd /content/up && find . -type f ! -name '*.c' ! -name '*.h' ! -name '*.cpp' ! -name '*.hpp' ! -path './.git/*' -exec cp --parents {{}} /content/n/{language}/ \\;")
        shutil.move(f"/content/n/{language}", src)
text = "".join(p.read_text(errors="ignore") for p in pathlib.Path(src).rglob("*") if p.suffix in (".c", ".h", ".cpp") and p.is_file())
lat = sum(c.isascii() and c.isalpha() for c in text); nat = sum((not c.isascii()) and c.isalpha() for c in text)
print(f"In {language}: {100 * nat // max(1, lat + nat)}% of the letters of the C and C++ source.")
print("Lowering through the original Hindawi pipeline, then building with CUDA where there is a GPU ...")
sh(f"rm -rf /content/llama-built && python3 {H}/lang/hindawi/tools/build_project.py {src} --lang {language} --work /content/llama-built --lower-only --cxx")
cuda = "ON" if shutil.which("nvcc") or os.path.exists("/usr/local/cuda/bin/nvcc") else "OFF"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
sh(f"cd /content/llama-built && cmake -B build -DLLAMA_CURL=OFF -DGGML_CUDA={cuda} >/dev/null && cmake --build build -j$(nproc) --target llama-cli >/dev/null")
print("Ready." if os.path.exists("/content/llama-built/build/bin/llama-cli") else "The build did not finish: see the messages above.")


# Model

In [ ]:
#@title **Choose a model** (downloaded once)
model = "Qwen2.5 7B (about 5.5 GB)" #@param ["Qwen2.5 0.5B (CPU or any GPU)", "Qwen2.5 3B (about 2.5 GB of GPU memory)", "Qwen2.5 7B (about 5.5 GB)", "Llama 3.1 8B (about 6 GB)", "Qwen2.5 14B (about 10 GB, a T4 or larger)"]
URLS = {"Qwen2.5 0.5B (CPU or any GPU)": "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf", "Qwen2.5 3B (about 2.5 GB of GPU memory)": "https://huggingface.co/Qwen/Qwen2.5-3B-Instruct-GGUF/resolve/main/qwen2.5-3b-instruct-q4_k_m.gguf", "Qwen2.5 7B (about 5.5 GB)": "https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf", "Llama 3.1 8B (about 6 GB)": "https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf", "Qwen2.5 14B (about 10 GB, a T4 or larger)": "https://huggingface.co/bartowski/Qwen2.5-14B-Instruct-GGUF/resolve/main/Qwen2.5-14B-Instruct-Q4_K_M.gguf"}
import os, subprocess
os.makedirs("/content/models", exist_ok=True); f = "/content/models/" + URLS[model].split("/")[-1]
if not os.path.exists(f): subprocess.run(["curl", "-fL", "-o", f, URLS[model]])
print(f, round(os.path.getsize(f) / 2**30, 2), "GB")


# Ask it

In [ ]:
#@title **Ask it** (the model runs inside the llama.cpp you built from your language)
prompt = "Explain in three sentences how a compiler turns a program into machine code." #@param {type:"string"}
tokens = 200 #@param {type:"integer"}
import subprocess, shutil
gpu = "99" if shutil.which("nvidia-smi") else "0"
r = subprocess.run(["/content/llama-built/build/bin/llama-cli", "-m", f, "-ngl", gpu, "-p", prompt, "-n", str(tokens), "--temp", "0.7", "-no-cnv", "--no-display-prompt"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])


Parashakti publishes llama.cpp in each language at github.com/hmnsq/llama.cpp (one branch per language). © 1993–2026 Abhishek Choudhary. All rights reserved. AyeAI.